# ⚙️ PredMarket Arb — Feature Engineering & Signal Detection

**Goal:** Answer one question: *do our features actually predict 
price direction in the next 15 minutes?*

| Result | Next step |
|--------|-----------|
| ✅ accuracy > 51%, 3+ strong features | Proceed to `train.py` |
| ⚠️ accuracy 50-51%, weak features | Redesign features first |
| ❌ accuracy = 50%, no separation | Stop — no signal exists |

Run all cells top to bottom. Takes ~3 minutes to complete.

> Ejecuta el notebook con **cwd en la raíz del repo** para que `data/raw/...` resuelva bien.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("husl")
pd.set_option('display.float_format', '{:.6f}'.format)

# ── CONFIG ────────────────────────────────────────────
ASSET    = "BTCUSDT"   # 💡 Change to ETHUSDT, SOLUSDT etc to explore others
DATA_DIR = Path("data/raw")
START    = "2021-01-01"  # Only modern market structure

# Target: predict direction N candles ahead (1 candle = 5min)
TARGET_CANDLES_AHEAD = 3   # 3 × 5min = 15min prediction horizon

FEATURES = [
    "ret_1", "ret_3", "ret_6", "ret_12", "mom_accel",
    "vol_10", "vol_20", "vol_ratio", "atr_5", "parkinson",
    "vol_zscore", "vol_trend",
    "hour", "dow", "is_ny_open"
]

print(f"✅ Config loaded — analyzing: {ASSET}")
print(f"   Prediction horizon: {TARGET_CANDLES_AHEAD * 5} minutes ahead")


## 1️⃣ Build Features From Raw 1min Data

We resample 1min → 5min because:
- Polymarket markets are 5min/15min windows
- Less noise than 1min
- Still enough resolution for momentum features


In [ ]:
# ── LOAD ──────────────────────────────────────────────
path = DATA_DIR / f"{ASSET}_1min.parquet"
assert path.exists(), f"❌ {path} not found — run download_datasets.py first"

df = pd.read_parquet(path).set_index("timestamp").sort_index()
df = df[df.index >= START]
print(f"Loaded {len(df):,} rows from {df.index.min()} to {df.index.max()}")

# ── RESAMPLE 1min → 5min ──────────────────────────────
df5 = df.resample("5T").agg({
    "open":   "first",
    "high":   "max",
    "low":    "min",
    "close":  "last",
    "volume": "sum"
}).dropna()

# ── MOMENTUM ──────────────────────────────────────────
df5["ret_1"]     = df5["close"].pct_change(1)
df5["ret_3"]     = df5["close"].pct_change(3)
df5["ret_6"]     = df5["close"].pct_change(6)
df5["ret_12"]    = df5["close"].pct_change(12)
df5["mom_accel"] = df5["ret_1"] - df5["ret_3"]

# ── VOLATILITY ────────────────────────────────────────
df5["vol_10"]    = df5["ret_1"].rolling(10).std()
df5["vol_20"]    = df5["ret_1"].rolling(20).std()
df5["vol_ratio"] = df5["vol_10"] / df5["vol_20"]
df5["atr_5"]     = (df5["high"] - df5["low"]).rolling(5).mean()
df5["parkinson"] = np.sqrt(
    (np.log(df5["high"] / df5["low"]) ** 2).rolling(10).mean()
    / (4 * np.log(2))
)

# ── VOLUME ────────────────────────────────────────────
vol_mean = df5["volume"].rolling(20).mean()
vol_std  = df5["volume"].rolling(20).std()
df5["vol_zscore"] = (df5["volume"] - vol_mean) / vol_std
df5["vol_trend"]  = df5["volume"] / df5["volume"].shift(3)

# ── TIME ──────────────────────────────────────────────
df5["hour"]       = df5.index.hour
df5["dow"]        = df5.index.dayofweek
df5["is_ny_open"] = ((df5["hour"] >= 13) & (df5["hour"] <= 16)).astype(int)

# ── TARGET ────────────────────────────────────────────
df5["target"] = (
    df5["close"].shift(-TARGET_CANDLES_AHEAD) >= df5["close"]
).astype(int)

# ── CLEAN ─────────────────────────────────────────────
df5 = df5.dropna(subset=FEATURES + ["target"])

print(f"\n✅ Features built")
print(f"   Rows:    {len(df5):,}")
print(f"   Features: {len(FEATURES)}")
print(f"   Target:  {df5.target.mean():.1%} UP / "
      f"{1-df5.target.mean():.1%} DOWN")
print(f"\nFirst 3 rows:")
df5[FEATURES[:6]].head(3)


## 2️⃣ Feature Distributions — Do they look healthy?

A healthy feature has:
- No extreme outliers (isolated spikes)  
- A recognizable distribution shape  
- Not 90%+ of values at exactly zero

**Red flag:** a feature that looks like a spike at 0 is useless.


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    lo, hi = df5[feat].quantile([0.01, 0.99])
    data = df5[feat].clip(lo, hi)
    axes[i].hist(data, bins=80, alpha=0.85, edgecolor='none', color='steelblue')
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_yticks([])
    axes[i].grid(alpha=0.2)
    # Show mean and std
    axes[i].axvline(data.mean(), color='yellow', linewidth=1.5,
                    linestyle='--', alpha=0.8)

plt.suptitle(
    "Feature Distributions  |  Yellow line = mean  |  "
    "Look for spikes at zero or extreme skew",
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

# 💡 TIP: If a feature is mostly zero it won't help the model.
# The yellow mean line should be near the center of the distribution.


## 3️⃣ Correlation Matrix — Are we doubling up on information?

High correlation (> 0.85) between two features means they carry
almost the same information. Keeping both wastes model capacity.


In [ ]:
corr = df5[FEATURES].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.zeros_like(corr, dtype=bool)
np.fill_diagonal(mask, True)   # hide diagonal (always 1.0)
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdYlGn",
    vmin=-1, vmax=1, ax=ax, linewidths=0.3,
    square=True, annot_kws={"size": 7}, mask=mask
)
ax.set_title("Feature Correlation Matrix\n"
             "Green = positive  |  Red = negative  |  "
             "Dark = strong correlation",
             fontsize=13, pad=20)
plt.tight_layout()
plt.show()

# Auto-flag redundant pairs
print("Correlation check:")
found = False
for i in range(len(FEATURES)):
    for j in range(i+1, len(FEATURES)):
        c = abs(corr.iloc[i, j])
        if c > 0.85:
            print(f"  ⚠️  {FEATURES[i]} ↔ {FEATURES[j]}: {c:.3f} "
                  f"— consider removing {FEATURES[j]}")
            found = True
if not found:
    print("  ✅ No highly correlated pairs — all features carry unique info")

# 💡 TIP: vol_10 and vol_20 will likely be correlated.
# That's OK — vol_ratio (their ratio) captures the regime change.


## 4️⃣ THE KEY QUESTION — Does each feature separate UP from DOWN?

For each feature we plot two distributions:
- 🟢 **Green** = values when BTC went UP in the next 15min
- 🔴 **Red** = values when BTC went DOWN in the next 15min

**Separated curves = predictive signal ✅**  
**Overlapping curves = useless feature ❌**

The KS statistic measures how different the two distributions are:
- `KS > 0.05` → strong signal 🟢
- `KS 0.02–0.05` → weak signal 🟡  
- `KS < 0.02` → no signal 🔴


In [ ]:
up   = df5[df5.target == 1]
down = df5[df5.target == 0]

fig, axes = plt.subplots(3, 5, figsize=(20, 11))
axes = axes.flatten()

signal_scores = {}
for i, feat in enumerate(FEATURES):
    lo, hi = df5[feat].quantile([0.01, 0.99])
    bins = np.linspace(lo, hi, 60)

    axes[i].hist(up[feat].clip(lo, hi),   bins=bins, alpha=0.55,
                 color='lime',   label='UP',   density=True)
    axes[i].hist(down[feat].clip(lo, hi), bins=bins, alpha=0.55,
                 color='red',    label='DOWN', density=True)

    ks_stat, p_val = stats.ks_2samp(
        up[feat].dropna(), down[feat].dropna()
    )
    signal_scores[feat] = ks_stat

    title_color = ('lime' if ks_stat > 0.05
                   else ('yellow' if ks_stat > 0.02 else 'tomato'))
    sig_label = ('★ STRONG' if ks_stat > 0.05
                 else ('~ weak' if ks_stat > 0.02 else '✗ none'))
    axes[i].set_title(f"{feat}\nKS={ks_stat:.3f}  {sig_label}",
                      fontsize=8.5, color=title_color)
    axes[i].set_yticks([])
    axes[i].grid(alpha=0.15)

axes[0].legend(fontsize=9, loc='upper right')
plt.suptitle(
    "UP 🟢 vs DOWN 🔴 distributions per feature\n"
    "Separated curves = predictive signal | "
    "KS statistic = separation score",
    fontsize=13, y=1.03
)
plt.tight_layout()
plt.show()


## 5️⃣ Signal Ranking — Best features at a glance


In [ ]:
signal_df = (
    pd.DataFrame.from_dict(signal_scores, orient='index',
                           columns=['KS'])
    .sort_values('KS', ascending=True)
)

colors = [
    'lime'   if v > 0.05 else
    'yellow' if v > 0.02 else
    'tomato'
    for v in signal_df.KS
]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(signal_df.index, signal_df.KS,
               color=colors, alpha=0.85, edgecolor='none')
ax.axvline(0.05, color='lime',   linestyle='--',
           alpha=0.7, label='Strong (0.05)')
ax.axvline(0.02, color='yellow', linestyle='--',
           alpha=0.7, label='Weak (0.02)')
ax.set_title("Feature Signal Strength (KS Statistic)\n"
             "Higher = better separation between UP and DOWN",
             fontsize=13)
ax.set_xlabel("KS Statistic")
ax.legend(fontsize=10)
ax.grid(alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

# Text summary
print("\n📊 Signal Ranking:")
print(f"{'Feature':<20} {'KS':>6}  {'Signal'}")
print("─" * 42)
for feat, row in signal_df.sort_values('KS', ascending=False).iterrows():
    ks   = row.KS
    icon = "✅" if ks > 0.05 else ("⚠️ " if ks > 0.02 else "❌")
    bar  = "█" * max(1, int(ks * 300))
    print(f"  {icon} {feat:<18} {ks:.4f}  {bar}")


## 6️⃣ Autocorrelation — Does the market have memory?

If returns are autocorrelated → past price action predicts future.  
If near zero → market is close to a random walk at this timeframe.

**Volatility clustering** (abs returns ACF) is almost always 
positive in crypto — high vol periods cluster together.  
This is exploitable: increase position size in high-vol regimes.


In [ ]:
from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df5["ret_1"].dropna().iloc[-10_000:]

autocorrelation_plot(sample, ax=axes[0])
axes[0].set_title("Return Autocorrelation\n"
                  "Significant lag → momentum exists")
axes[0].set_xlim(0, 50)
axes[0].set_ylim(-0.1, 0.1)
axes[0].axhline(0.02,  color='lime',   linestyle=':', alpha=0.6)
axes[0].axhline(-0.02, color='tomato', linestyle=':', alpha=0.6)
axes[0].grid(alpha=0.25)

autocorrelation_plot(sample.abs(), ax=axes[1])
axes[1].set_title("Absolute Return Autocorrelation\n"
                  "Positive = volatility clusters (normal in crypto)")
axes[1].set_xlim(0, 50)
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

# 💡 TIP: Even a tiny ACF at lag-1 (e.g. 0.03) is real signal
# at scale. The model will find and exploit it automatically.


## 7️⃣ Best Hours To Trade — When does the signal work?

Not all hours are equal. During low-liquidity hours:
- Spreads are wider on Polymarket  
- BTC barely moves → near-resolution edge disappears  
- Model accuracy drops  

We want to operate **only during high-signal hours**.


In [ ]:
hourly = df5.groupby("hour").agg(
    avg_volatility  = ("vol_10",  "mean"),
    predictability  = ("target",  lambda x: abs(x.mean() - 0.5) * 2),
    avg_vol_zscore  = ("vol_zscore", "mean"),
    candle_count    = ("target",  "count")
).round(4)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].bar(hourly.index, hourly.avg_volatility,
            alpha=0.85, color='cyan')
axes[0].set_title("Avg Volatility by Hour UTC", fontsize=12)
axes[0].set_xlabel("Hour UTC")
axes[0].axvspan(13, 16, alpha=0.12, color='yellow')

axes[1].bar(hourly.index, hourly.predictability,
            alpha=0.85, color='lime')
axes[1].set_title("Directional Predictability\n(higher = less random)",
                  fontsize=12)
axes[1].set_xlabel("Hour UTC")
axes[1].axvspan(13, 16, alpha=0.12, color='yellow')

axes[2].bar(hourly.index, hourly.candle_count,
            alpha=0.85, color='orange')
axes[2].set_title("Data Points by Hour", fontsize=12)
axes[2].set_xlabel("Hour UTC")

for ax in axes:
    ax.grid(alpha=0.2)
    ax.set_xticks(range(0, 24, 2))

plt.suptitle("Yellow band = NY Open (13-16 UTC)", fontsize=11)
plt.tight_layout()
plt.show()

best_hours = (
    hourly
    .nlargest(6, 'predictability')
    .sort_index()
    .index
    .tolist()
)
print(f"\n⏰ Best hours to operate (UTC): {best_hours}")
print("   → Save these in config.py as ACTIVE_HOURS")
worst_h = int(hourly.predictability.idxmin())
print(
    f"\n   Worst hour: {worst_h}:00 UTC "
    f"(predictability={hourly.predictability.min():.4f})"
)


## 8️⃣ LightGBM — Holdout accuracy

Baseline gradient boosting on the same features. If accuracy is near 50%,
nonlinear interactions may still be weak — cross-check with the KS ranking above.


In [ ]:
X = df5[FEATURES]
y = df5["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_set = lgb.Dataset(X_train, label=y_train)
test_set = lgb.Dataset(X_test, label=y_test, reference=train_set)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": 42,
}

bst = lgb.train(
    params,
    train_set,
    num_boost_round=400,
    valid_sets=[test_set],
    callbacks=[lgb.early_stopping(40, verbose=False)],
)

proba = bst.predict(X_test, num_iteration=bst.best_iteration)
pred = (proba >= 0.5).astype(int)
acc = float((pred == y_test).mean())

print(f"\n🎯 Holdout accuracy: {acc:.2%}")
print("   (random baseline = 50%)\n")

imp = pd.Series(dict(zip(FEATURES, bst.feature_importance(importance_type="gain"))))
imp = imp.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
imp.plot(kind="barh", ax=ax, color="steelblue", edgecolor="none")
ax.set_title("LightGBM feature importance (gain)", fontsize=12)
ax.grid(alpha=0.2, axis="x")
plt.tight_layout()
plt.show()


## 9️⃣ Calibration & final verdict

Calibration: predicted probabilities vs observed frequency of UP.
Use the intro table to decide whether to move on to `train.py` or redesign features.


In [ ]:
prob_true, prob_pred = calibration_curve(
    y_test, proba, n_bins=10, strategy="uniform"
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(prob_pred, prob_true, marker="o", linewidth=2, label="Model")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration (holdout)", fontsize=12)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

n_strong = sum(1 for v in signal_scores.values() if v > 0.05)
verdict = "✅"
next_step = "Proceed to train.py"
if acc <= 0.50:
    verdict, next_step = "❌", "Stop — no signal exists"
elif acc < 0.51 or n_strong < 3:
    verdict, next_step = "⚠️", "Redesign features first"

print("\n" + "=" * 52)
print(" FINAL VERDICT")
print("=" * 52)
print(f"  Holdout accuracy:          {acc:.2%}")
print(f"  Strong features (KS>0.05): {n_strong}")
print(f"  Verdict: {verdict}  →  {next_step}")
print("=" * 52 + "\n")
